# Niche DE Downstream Analysis — Pathway Enrichment & Biological Interpretation

**Aim 2 — Interpreting niche-driven differential expression results**

This notebook reads all DE results from the Niche-Driven DE pipeline and:

1. Examines top DE genes in priority contrasts (Approach B immune niches)
2. Checks overlap with hypothesis-driven gene signatures (exhaustion, checkpoint, CAF, immune evasion)
3. Generates signature heatmaps showing LFC across contrasts
4. Runs pathway enrichment (KEGG, GO, Hallmark, WikiPathways) via Enrichr and GSEA prerank
5. Compares DE genes across immune niches to find shared vs unique biology
6. Produces publication-ready volcano plots with signature gene highlighting

**Requires:** `gseapy` (`pip install gseapy`), DE results from `Niche_Driven_DE.ipynb`

## Imports and Setup

In [ ]:
#!/usr/bin/env python3
"""
Niche DE Downstream Analysis — Pathway Enrichment & Biological Interpretation
==============================================================================
Reads DE results from Niche_Driven_DE, examines top genes, checks for
exhaustion/immune evasion signature overlap, and runs pathway enrichment.

Requires: gseapy (pip install gseapy)
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import glob
import warnings
from datetime import datetime

warnings.filterwarnings('ignore')

## 1. CONFIGURATION

In [ ]:
DE_ROOT = "/Users/adiallo/Desktop/Thesis/Results/AIM_2_Results/Niche_DE_Analysis"
OUTPUT_DIR = "/Users/adiallo/Desktop/Thesis/Results/AIM_2_Results/Niche_DE_Downstream"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Significance thresholds
PADJ_THRESH = 0.05
LFC_THRESH = 1.0  # for volcano "significant" calls
LFC_THRESH_MILD = 0.5  # for pathway input (more inclusive)

# --- GENE SIGNATURES OF INTEREST ---

# CD8+ T-cell exhaustion — staged model
# Progenitor: PD-1+ TIM-3- Ly108+ CD69- TCF-1+
EXHAUSTION_PROGENITOR = [
    'TCF7',    # TCF-1, master TF for progenitor exhausted
    'BCL6',    # transcriptional repressor, maintains progenitor state
    'IL7R',    # CD127, survival signaling
    'CXCR5',   # follicular homing
    'SELL',    # CD62L, lymph node homing
    'LEF1',    # Wnt/TCF pathway, stemness
]

# Intermediate: PD-1+ TIM-3± Ly108± CD69+
EXHAUSTION_INTERMEDIATE = [
    'PDCD1',   # PD-1
    'HAVCR2',  # TIM-3
    'LAG3',    # LAG-3
    'TIGIT',   # checkpoint receptor
    'TOX',     # master TF for exhaustion program
    'BATF',    # AP-1 family, exhaustion-associated
    'EOMES',   # T-box TF, intermediate exhaustion
]

# Terminal: PD-1High TIM-3+ Ly108- CD69+ TCF-1-
EXHAUSTION_TERMINAL = [
    'IFNG',    # IFN-gamma, residual effector function
    'GZMB',    # Granzyme B, cytotoxicity
    'PRF1',    # Perforin
    'CXCL13',  # hallmark of terminally exhausted CD8
    'ENTPD1',  # CD39, terminal exhaustion marker
    'CTLA4',   # upregulated in terminal exhaustion
    'CD160',   # inhibitory receptor
    'CD244',   # 2B4, inhibitory receptor
    'LAYN',    # terminal exhaustion marker
]

# Transitional modifier (peaks during intermediate phase)
EXHAUSTION_TRANSITIONAL = [
    'TNFRSF18',  # GITR — transient activation/early exhaustion indicator
]

# Combined flat list for backward compatibility
EXHAUSTION_SIGNATURE = (EXHAUSTION_PROGENITOR + EXHAUSTION_INTERMEDIATE +
                        EXHAUSTION_TERMINAL + EXHAUSTION_TRANSITIONAL)

# Staged dictionary for heatmap grouping
EXHAUSTION_STAGED = {
    'Progenitor': EXHAUSTION_PROGENITOR,
    'Intermediate': EXHAUSTION_INTERMEDIATE,
    'Terminal': EXHAUSTION_TERMINAL,
    'Transitional': EXHAUSTION_TRANSITIONAL,
}

# Immune checkpoint genes
CHECKPOINT_GENES = [
    'PDCD1', 'CD274', 'PDCD1LG2',  # PD-1/PD-L1/PD-L2
    'CTLA4', 'CD80', 'CD86',        # CTLA-4 axis
    'LAG3', 'HAVCR2', 'TIGIT',      # other checkpoints
    'VSIR', 'BTLA', 'IDO1', 'IDO2', # inhibitory
    'CD276', 'VTCN1',                # B7 family
]

# CAF-associated genes (relevant to your CAF-CD8 hypothesis)
CAF_GENES = [
    'FAP', 'ACTA2', 'PDGFRA', 'PDGFRB',      # canonical CAF markers
    'COL1A1', 'COL1A2', 'COL3A1', 'FN1',      # ECM
    'TGFB1', 'TGFB2', 'TGFB3',               # TGF-beta (immunosuppressive)
    'CXCL12', 'CXCL14',                        # CAF chemokines
    'IL6', 'IL11',                              # CAF cytokines
    'MMP2', 'MMP3', 'MMP9', 'MMP14',          # matrix remodeling
    'VEGFA', 'VEGFB',                           # angiogenesis
    'WNT2', 'WNT5A',                           # Wnt signaling
    'GREM1',                                    # BMP antagonist (GREM1+ CAF)
    'CCL2', 'CCL5',                             # immune cell recruitment
]

# Immune evasion / tumor immune escape genes
IMMUNE_EVASION_GENES = [
    'CD47', 'B2M', 'HLA-A', 'HLA-B', 'HLA-C',  # MHC-I / antigen presentation
    'HLA-E', 'HLA-F', 'HLA-G',                    # non-classical MHC
    'TAP1', 'TAP2', 'TAPBP',                      # antigen processing
    'IFNG', 'IFNGR1', 'IFNGR2',                   # IFN-gamma signaling
    'GZMA', 'GZMB', 'GZMK', 'PRF1',              # cytotoxicity
    'CMTM6', 'CD58',                               # from your proposal (Ho et al.)
    'STK11', 'SOX17',                              # therapeutic targets from your proposal
]

ALL_SIGNATURES = {
    'Exhaustion_Progenitor': EXHAUSTION_PROGENITOR,
    'Exhaustion_Intermediate': EXHAUSTION_INTERMEDIATE,
    'Exhaustion_Terminal': EXHAUSTION_TERMINAL,
    'Exhaustion_Transitional': EXHAUSTION_TRANSITIONAL,
    'Exhaustion_All': EXHAUSTION_SIGNATURE,
    'Checkpoint': CHECKPOINT_GENES,
    'CAF_associated': CAF_GENES,
    'Immune_evasion': IMMUNE_EVASION_GENES,
}

## 2. LOAD ALL DE RESULTS

In [ ]:
print("=" * 70)
print("Loading all DE results")
print("=" * 70)

de_results = {}

for approach_dir in sorted(glob.glob(os.path.join(DE_ROOT, "Approach_*"))):
    approach_name = os.path.basename(approach_dir)

    # Handle Approach C subdirectories
    de_files = glob.glob(os.path.join(approach_dir, "DE_*.csv"))
    if len(de_files) == 0:
        # Check subdirectories (Approach C has threshold_median, threshold_Q75)
        for sub in sorted(glob.glob(os.path.join(approach_dir, "threshold_*"))):
            sub_name = os.path.basename(sub)
            for f in sorted(glob.glob(os.path.join(sub, "DE_*.csv"))):
                contrast_name = os.path.basename(f).replace("DE_", "").replace(".csv", "")
                key = f"{approach_name}/{sub_name}/{contrast_name}"
                try:
                    df = pd.read_csv(f, index_col=0)
                    de_results[key] = df
                except:
                    pass
    else:
        for f in sorted(de_files):
            contrast_name = os.path.basename(f).replace("DE_", "").replace(".csv", "")
            key = f"{approach_name}/{contrast_name}"
            try:
                df = pd.read_csv(f, index_col=0)
                de_results[key] = df
            except:
                pass

print(f"Loaded {len(de_results)} DE result tables\n")

for key in sorted(de_results.keys()):
    df = de_results[key]
    n_sig = (df['padj'] < PADJ_THRESH).sum() if 'padj' in df.columns else 0
    print(f"  {key}: {len(df)} genes, {n_sig} significant")

## 3. EXAMINE TOP DE GENES — PRIORITY CONTRASTS

In [ ]:
print(f"\n{'=' * 70}")
print("Top DE genes in priority contrasts")
print("=" * 70)

# Define priority contrasts (Approach B immune niches, mets comparisons)
priority_keys = [k for k in de_results.keys()
                 if 'Approach_B' in k and ('Distant_vs_No__CT' in k or 'LN_vs_No__CT' in k)]

# Also include Approach A mets contrasts with signal
priority_keys += [k for k in de_results.keys()
                  if 'Approach_A' in k and 'Distant_vs_No__CT' in k
                  and (de_results[k]['padj'] < PADJ_THRESH).sum() > 10]

top_genes_all = {}

for key in sorted(priority_keys):
    df = de_results[key]
    sig = df[df['padj'] < PADJ_THRESH].copy()

    if len(sig) == 0:
        continue

    sig_up = sig[sig['log2FoldChange'] > 0].sort_values('padj')
    sig_down = sig[sig['log2FoldChange'] < 0].sort_values('padj')

    print(f"\n--- {key} ---")
    print(f"  Total significant: {len(sig)} (↑{len(sig_up)} ↓{len(sig_down)})")

    print(f"\n  Top 15 upregulated:")
    for gene in sig_up.head(15).index:
        row = sig_up.loc[gene]
        print(f"    {gene:20s}  LFC={row['log2FoldChange']:+.3f}  padj={row['padj']:.2e}")

    print(f"\n  Top 15 downregulated:")
    for gene in sig_down.head(15).index:
        row = sig_down.loc[gene]
        print(f"    {gene:20s}  LFC={row['log2FoldChange']:+.3f}  padj={row['padj']:.2e}")

    top_genes_all[key] = {
        'up': sig_up.head(50).index.tolist(),
        'down': sig_down.head(50).index.tolist(),
        'all_sig': sig.index.tolist(),
    }

## 4. SIGNATURE GENE OVERLAP ANALYSIS

In [ ]:
print(f"\n{'=' * 70}")
print("Signature gene overlap analysis")
print("=" * 70)

signature_overlap_results = []

for key in sorted(priority_keys):
    df = de_results[key]
    sig = df[df['padj'] < PADJ_THRESH]

    if len(sig) == 0:
        continue

    print(f"\n--- {key} ({len(sig)} DE genes) ---")

    for sig_name, sig_genes in ALL_SIGNATURES.items():
        # Check which signature genes are in the DE results at all
        present = [g for g in sig_genes if g in df.index]
        # Which are significant
        sig_overlap = [g for g in sig_genes if g in sig.index]
        # Direction
        if len(sig_overlap) > 0:
            up = [g for g in sig_overlap if df.loc[g, 'log2FoldChange'] > 0]
            down = [g for g in sig_overlap if df.loc[g, 'log2FoldChange'] < 0]
        else:
            up, down = [], []

        if len(sig_overlap) > 0:
            print(f"\n  {sig_name}: {len(sig_overlap)}/{len(present)} genes significant")
            for g in sig_overlap:
                row = df.loc[g]
                direction = "↑" if row['log2FoldChange'] > 0 else "↓"
                print(f"    {direction} {g:15s}  LFC={row['log2FoldChange']:+.3f}  padj={row['padj']:.2e}")

        signature_overlap_results.append({
            'contrast': key,
            'signature': sig_name,
            'n_tested': len(present),
            'n_significant': len(sig_overlap),
            'n_up': len(up),
            'n_down': len(down),
            'genes_up': '; '.join(up),
            'genes_down': '; '.join(down),
        })

# Save overlap results
overlap_df = pd.DataFrame(signature_overlap_results)
overlap_df.to_csv(os.path.join(OUTPUT_DIR, 'signature_overlap_results.csv'), index=False)
print(f"\nSaved signature overlap to: signature_overlap_results.csv")

## 5. SIGNATURE HEATMAP — LFC of signature genes across contrasts

In [ ]:
print(f"\n{'=' * 70}")
print("Generating signature heatmaps")
print("=" * 70)

for sig_name, sig_genes in ALL_SIGNATURES.items():
    # Build LFC matrix: genes × contrasts
    lfc_data = {}
    padj_data = {}

    for key in sorted(priority_keys):
        df = de_results[key]
        short_key = key.split('/')[-1]  # just the contrast name

        lfcs = []
        padjs = []
        valid_genes = []
        for g in sig_genes:
            if g in df.index:
                lfcs.append(df.loc[g, 'log2FoldChange'])
                padjs.append(df.loc[g, 'padj'])
                valid_genes.append(g)

        if len(valid_genes) > 0:
            lfc_data[short_key] = pd.Series(lfcs, index=valid_genes)
            padj_data[short_key] = pd.Series(padjs, index=valid_genes)

    if len(lfc_data) == 0:
        continue

    lfc_matrix = pd.DataFrame(lfc_data)
    padj_matrix = pd.DataFrame(padj_data)

    # Only include genes present in at least one contrast
    lfc_matrix = lfc_matrix.dropna(how='all')
    if len(lfc_matrix) < 2:
        continue

    # Create annotation for significance
    annot = lfc_matrix.round(2).astype(str)
    for col in padj_matrix.columns:
        for gene in padj_matrix.index:
            if gene in annot.index and col in annot.columns:
                if pd.notna(padj_matrix.loc[gene, col]) and padj_matrix.loc[gene, col] < 0.05:
                    annot.loc[gene, col] = annot.loc[gene, col] + " *"

    fig, ax = plt.subplots(figsize=(max(10, len(lfc_data) * 2.5), max(6, len(lfc_matrix) * 0.4)),
                           facecolor='white')

    vmax = max(abs(lfc_matrix.min().min()), abs(lfc_matrix.max().max()))
    vmax = min(vmax, 4)  # cap for readability

    sns.heatmap(lfc_matrix, cmap='RdBu_r', center=0, vmin=-vmax, vmax=vmax,
                annot=annot, fmt='', annot_kws={'fontsize': 8},
                linewidths=0.5, linecolor='white', ax=ax,
                cbar_kws={'label': 'log2 Fold Change', 'shrink': 0.8})

    ax.set_title(f'{sig_name} Signature — log2FC across contrasts\n(* = padj < 0.05)',
                 fontsize=13, fontweight='bold')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=9)
    ax.set_yticklabels(ax.get_yticklabels(), fontsize=10)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, f'heatmap_{sig_name}_signature.png'),
                dpi=300, bbox_inches='tight')
    plt.show()

print("Signature heatmaps saved.")

## 6. GENE SET ENRICHMENT — KEGG, GO, Hallmark via gseapy

In [ ]:
print(f"\n{'=' * 70}")
print("Running pathway enrichment analysis (gseapy)")
print("=" * 70)

try:
    import gseapy as gp
    HAS_GSEAPY = True
    print("gseapy loaded successfully")
except ImportError:
    HAS_GSEAPY = False
    print("WARNING: gseapy not installed. Run: pip install gseapy")
    print("Skipping enrichment analysis.")

if HAS_GSEAPY:

    # Gene set libraries to query
    GENE_SET_LIBS = [
        'KEGG_2021_Human',
        'GO_Biological_Process_2023',
        'MSigDB_Hallmark_2020',
        'WikiPathway_2023_Human',
    ]

    enrichment_results_all = []

    for key in sorted(priority_keys):
        df = de_results[key]
        sig = df[df['padj'] < PADJ_THRESH].copy()

        if len(sig) < 5:
            continue

        short_key = key.replace('/', '__')
        contrast_out = os.path.join(OUTPUT_DIR, 'enrichment', short_key)
        os.makedirs(contrast_out, exist_ok=True)

        print(f"\n--- {key} ({len(sig)} DE genes) ---")

        # Split into up and down
        sig_up = sig[sig['log2FoldChange'] > 0].index.tolist()
        sig_down = sig[sig['log2FoldChange'] < 0].index.tolist()

        for direction, gene_list in [('up', sig_up), ('down', sig_down), ('all', sig.index.tolist())]:
            if len(gene_list) < 5:
                continue

            print(f"  {direction}: {len(gene_list)} genes")

            for lib in GENE_SET_LIBS:
                try:
                    enr = gp.enrichr(
                        gene_list=gene_list,
                        gene_sets=lib,
                        organism='human',
                        outdir=None,
                        no_plot=True,
                    )

                    res = enr.results
                    res_sig = res[res['Adjusted P-value'] < 0.05].head(20)

                    if len(res_sig) > 0:
                        print(f"    {lib}: {len(res_sig)} significant terms")
                        for _, row in res_sig.head(5).iterrows():
                            print(f"      {row['Term'][:60]:60s}  padj={row['Adjusted P-value']:.2e}")

                        # Save
                        res_sig['contrast'] = key
                        res_sig['direction'] = direction
                        res_sig['library'] = lib
                        enrichment_results_all.append(res_sig)

                        res.to_csv(os.path.join(contrast_out, f'{direction}_{lib}.csv'), index=False)

                except Exception as e:
                    print(f"    {lib}: FAILED — {e}")

        # --- Prerank GSEA (uses full ranked gene list) ---
        print(f"  Running prerank GSEA...")
        try:
            ranked = df[['log2FoldChange']].dropna().copy()
            ranked = ranked[~ranked.index.duplicated()]
            ranked = ranked.sort_values('log2FoldChange', ascending=False)

            for lib in ['KEGG_2021_Human', 'MSigDB_Hallmark_2020']:
                try:
                    pre_res = gp.prerank(
                        rnk=ranked,
                        gene_sets=lib,
                        threads=4,
                        min_size=15,
                        max_size=500,
                        permutation_num=1000,
                        outdir=None,
                        seed=42,
                        no_plot=True,
                    )

                    gsea_sig = pre_res.res2d[pre_res.res2d['FDR q-val'] < 0.25].head(20)
                    if len(gsea_sig) > 0:
                        print(f"    Prerank {lib}: {len(gsea_sig)} terms (FDR<0.25)")
                        for _, row in gsea_sig.head(5).iterrows():
                            direction_str = "↑" if row['NES'] > 0 else "↓"
                            print(f"      {direction_str} {row['Term'][:55]:55s}  NES={row['NES']:.2f}  FDR={row['FDR q-val']:.3f}")

                    gsea_sig['contrast'] = key
                    gsea_sig['library'] = lib
                    gsea_sig['method'] = 'prerank'
                    enrichment_results_all.append(gsea_sig)

                    pre_res.res2d.to_csv(os.path.join(contrast_out, f'prerank_{lib}.csv'), index=False)

                except Exception as e:
                    print(f"    Prerank {lib}: FAILED — {e}")

        except Exception as e:
            print(f"  Prerank setup FAILED: {e}")

    # Combine all enrichment results
    if enrichment_results_all:
        combined_enr = pd.concat(enrichment_results_all, ignore_index=True)
        combined_enr.to_csv(os.path.join(OUTPUT_DIR, 'all_enrichment_results.csv'), index=False)
        print(f"\nSaved combined enrichment results: {len(combined_enr)} rows")

## 7. CROSS-NICHE COMPARISON — Shared and unique DE genes

In [ ]:
print(f"\n{'=' * 70}")
print("Cross-niche DE gene comparison")
print("=" * 70)

# Compare Approach B niches: what's shared between immune niches?
approach_b_ct_distant = {k: de_results[k] for k in de_results
                         if 'Approach_B' in k and 'Distant_vs_No__CT' in k}

if len(approach_b_ct_distant) >= 2:
    print("\nApproach B — Distant_vs_No in CT: shared DE genes across immune niches")

    sig_gene_sets = {}
    for key, df in approach_b_ct_distant.items():
        niche_name = key.split('/')[0].replace('Approach_B_', '')
        sig = df[df['padj'] < PADJ_THRESH]
        sig_gene_sets[niche_name] = set(sig.index)
        print(f"  {niche_name}: {len(sig)} DE genes")

    # Pairwise overlaps
    niche_names = list(sig_gene_sets.keys())
    print("\n  Pairwise overlaps:")
    for i in range(len(niche_names)):
        for j in range(i+1, len(niche_names)):
            n1, n2 = niche_names[i], niche_names[j]
            shared = sig_gene_sets[n1] & sig_gene_sets[n2]
            print(f"    {n1} ∩ {n2}: {len(shared)} shared genes")
            if len(shared) > 0 and len(shared) <= 30:
                print(f"      {', '.join(sorted(shared)[:20])}")

    # Genes shared across ALL immune niches
    if len(sig_gene_sets) >= 2:
        all_shared = set.intersection(*sig_gene_sets.values())
        print(f"\n  Genes shared across ALL immune niches: {len(all_shared)}")
        if 0 < len(all_shared) <= 50:
            for g in sorted(all_shared):
                lfcs = []
                for key, df in approach_b_ct_distant.items():
                    if g in df.index:
                        lfcs.append(f"{df.loc[g, 'log2FoldChange']:+.2f}")
                print(f"    {g:20s}  LFCs: {', '.join(lfcs)}")

    # Save
    overlap_summary = []
    for name, genes in sig_gene_sets.items():
        for g in genes:
            overlap_summary.append({'niche': name, 'gene': g})
    pd.DataFrame(overlap_summary).to_csv(
        os.path.join(OUTPUT_DIR, 'cross_niche_DE_genes.csv'), index=False)

## 8. COMBINED VOLCANO PLOTS — Priority contrasts side by side

In [ ]:
print(f"\n{'=' * 70}")
print("Generating combined volcano plots")
print("=" * 70)

def make_volcano_enhanced(ax, res_df, title_str, highlight_genes=None):
    """Volcano plot with optional gene highlighting."""
    res = res_df.dropna(subset=["padj", "log2FoldChange"])
    if len(res) == 0:
        ax.set_title(f"{title_str}\n(no data)")
        return

    sig_up = (res["padj"] < PADJ_THRESH) & (res["log2FoldChange"] > LFC_THRESH)
    sig_down = (res["padj"] < PADJ_THRESH) & (res["log2FoldChange"] < -LFC_THRESH)
    ns = ~(sig_up | sig_down)

    ax.scatter(res.loc[ns, "log2FoldChange"], -np.log10(res.loc[ns, "padj"]),
               c="gray", alpha=0.2, s=4, zorder=1)
    ax.scatter(res.loc[sig_up, "log2FoldChange"], -np.log10(res.loc[sig_up, "padj"]),
               c="#D62728", alpha=0.5, s=8, label=f"Up ({sig_up.sum()})", zorder=2)
    ax.scatter(res.loc[sig_down, "log2FoldChange"], -np.log10(res.loc[sig_down, "padj"]),
               c="#1F77B4", alpha=0.5, s=8, label=f"Down ({sig_down.sum()})", zorder=2)

    # Highlight signature genes
    if highlight_genes:
        for gene_list, color, label in highlight_genes:
            in_data = [g for g in gene_list if g in res.index]
            if len(in_data) > 0:
                ax.scatter(res.loc[in_data, "log2FoldChange"],
                          -np.log10(res.loc[in_data, "padj"]),
                          c=color, s=40, edgecolors='black', linewidths=0.5,
                          label=label, zorder=3)
                for g in in_data:
                    if res.loc[g, "padj"] < PADJ_THRESH:
                        ax.annotate(g, (res.loc[g, "log2FoldChange"],
                                       -np.log10(res.loc[g, "padj"])),
                                   fontsize=6, alpha=0.9, fontweight='bold')

    ax.axhline(-np.log10(PADJ_THRESH), color="black", linestyle=":", alpha=0.3)
    ax.axvline(LFC_THRESH, color="black", linestyle=":", alpha=0.3)
    ax.axvline(-LFC_THRESH, color="black", linestyle=":", alpha=0.3)
    ax.set_xlabel("log2 Fold Change", fontsize=10)
    ax.set_ylabel("-log10(padj)", fontsize=10)
    ax.set_title(title_str, fontsize=10, fontweight='bold')
    ax.legend(fontsize=7, loc='upper left')


# Approach B: immune niches — Distant_vs_No in CT
b_contrasts = [(k, de_results[k]) for k in sorted(de_results.keys())
               if 'Approach_B' in k and 'Distant_vs_No__CT' in k]

if b_contrasts:
    fig, axes = plt.subplots(1, len(b_contrasts), figsize=(7 * len(b_contrasts), 7))
    if len(b_contrasts) == 1:
        axes = [axes]

    highlight = [
        (EXHAUSTION_SIGNATURE, '#FF6B00', 'Exhaustion'),
        (CHECKPOINT_GENES, '#9B59B6', 'Checkpoint'),
    ]

    for ax, (key, df) in zip(axes, b_contrasts):
        niche_name = key.split('/')[0].replace('Approach_B_', '').replace('_', ' ')
        make_volcano_enhanced(ax, df, f"{niche_name}\nDistant Mets vs No Mets (CT)",
                             highlight_genes=highlight)

    plt.suptitle("Immune Niches — Distant Mets vs No Mets in Tumor Core",
                 fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'volcanos_approach_B_distant_vs_no_CT.png'),
                dpi=300, bbox_inches='tight')
    plt.show()

# Same for LN_vs_No
b_ln = [(k, de_results[k]) for k in sorted(de_results.keys())
        if 'Approach_B' in k and 'LN_vs_No__CT' in k]

if b_ln:
    fig, axes = plt.subplots(1, len(b_ln), figsize=(7 * len(b_ln), 7))
    if len(b_ln) == 1:
        axes = [axes]

    for ax, (key, df) in zip(axes, b_ln):
        niche_name = key.split('/')[0].replace('Approach_B_', '').replace('_', ' ')
        make_volcano_enhanced(ax, df, f"{niche_name}\nLN Mets vs No Mets (CT)",
                             highlight_genes=highlight)

    plt.suptitle("Immune Niches — LN Mets vs No Mets in Tumor Core",
                 fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'volcanos_approach_B_LN_vs_no_CT.png'),
                dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
print(f"\n{'=' * 70}")
print("Exhaustion signature detail — STAGED analysis")
print("=" * 70)

exh_detail = []
for key in sorted(de_results.keys()):
    df = de_results[key]
    for stage, genes in EXHAUSTION_STAGED.items():
        for gene in genes:
            if gene in df.index:
                row = df.loc[gene]
                exh_detail.append({
                    'contrast': key,
                    'stage': stage,
                    'gene': gene,
                    'log2FoldChange': row['log2FoldChange'],
                    'padj': row['padj'],
                    'baseMean': row.get('baseMean', np.nan),
                    'significant': row['padj'] < PADJ_THRESH if pd.notna(row['padj']) else False,
                })

exh_df = pd.DataFrame(exh_detail)
if len(exh_df) > 0:
    exh_df.to_csv(os.path.join(OUTPUT_DIR, 'exhaustion_staged_all_contrasts.csv'), index=False)

    # Filter to priority contrasts (Approach B, CT)
    priority_contrasts = [c for c in exh_df['contrast'].unique()
                          if 'Approach_B' in c and 'CT' in c]

    if len(priority_contrasts) > 0:
        exh_priority = exh_df[exh_df['contrast'].isin(priority_contrasts)]

        # Pivot: gene × contrast, with stage ordering
        exh_pivot = exh_priority.pivot_table(
            index='gene', columns='contrast',
            values='log2FoldChange', aggfunc='first')

        # Order genes by exhaustion stage
        stage_order = []
        for stage in ['Progenitor', 'Intermediate', 'Terminal', 'Transitional']:
            for gene in EXHAUSTION_STAGED[stage]:
                if gene in exh_pivot.index:
                    stage_order.append(gene)
        exh_pivot = exh_pivot.loc[stage_order]

        # Build stage labels for row coloring
        gene_to_stage = {}
        for stage, genes in EXHAUSTION_STAGED.items():
            for g in genes:
                gene_to_stage[g] = stage
        stage_colors_map = {
            'Progenitor': '#2ECC71',
            'Intermediate': '#F39C12',
            'Terminal': '#E74C3C',
            'Transitional': '#9B59B6',
        }

        # Build significance annotation
        padj_pivot = exh_priority.pivot_table(
            index='gene', columns='contrast',
            values='padj', aggfunc='first')
        padj_pivot = padj_pivot.loc[stage_order]

        annot = exh_pivot.round(2).astype(str)
        for col in padj_pivot.columns:
            for gene in padj_pivot.index:
                if gene in annot.index and col in annot.columns:
                    if pd.notna(padj_pivot.loc[gene, col]) and padj_pivot.loc[gene, col] < 0.05:
                        annot.loc[gene, col] = annot.loc[gene, col] + " *"

        # Clean column names
        clean_cols = []
        for c in exh_pivot.columns:
            parts = c.split('/')
            niche = parts[0].replace('Approach_B_', '').replace('Niche_', 'N')
            contrast = parts[-1].replace('_', ' ')
            clean_cols.append(f"{niche}\n{contrast}")

        # Plot
        fig, ax = plt.subplots(
            figsize=(max(10, len(exh_pivot.columns) * 3), max(6, len(exh_pivot) * 0.45)),
            facecolor='white')

        vmax = max(abs(exh_pivot.min().min()), abs(exh_pivot.max().max()))
        vmax = min(vmax, 3)

        sns.heatmap(exh_pivot, cmap='RdBu_r', center=0, vmin=-vmax, vmax=vmax,
                   annot=annot, fmt='', annot_kws={'fontsize': 8},
                   linewidths=0.5, linecolor='white', ax=ax,
                   cbar_kws={'label': 'log2 Fold Change', 'shrink': 0.7},
                   xticklabels=clean_cols)

        # Color y-axis labels by exhaustion stage
        for label in ax.get_yticklabels():
            gene = label.get_text()
            stage = gene_to_stage.get(gene, 'Unknown')
            label.set_color(stage_colors_map.get(stage, 'black'))
            label.set_fontweight('bold')
        ax.set_yticklabels(ax.get_yticklabels(), fontsize=10)
        ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=9)

        ax.set_title(
            'CD8+ T-cell Exhaustion Stages — log2FC in Immune Niches (CT)\n'
            '(* = padj < 0.05 | '
            'Green=Progenitor  Orange=Intermediate  Red=Terminal  Purple=Transitional)',
            fontsize=12, fontweight='bold')
        ax.set_ylabel('')

        plt.tight_layout()
        plt.savefig(os.path.join(OUTPUT_DIR, 'exhaustion_staged_heatmap.png'),
                   dpi=300, bbox_inches='tight')
        plt.show()

        # Print summary: which stages are shifting in metastatic patients?
        print("\n--- Exhaustion stage summary across Approach B contrasts ---")
        for contrast in priority_contrasts:
            c_data = exh_priority[exh_priority['contrast'] == contrast]
            short = contrast.split('/')[-1]
            print(f"\n  {short}:")
            for stage in ['Progenitor', 'Intermediate', 'Terminal', 'Transitional']:
                stage_data = c_data[c_data['stage'] == stage]
                sig_data = stage_data[stage_data['significant'] == True]
                if len(sig_data) > 0:
                    mean_lfc = sig_data['log2FoldChange'].mean()
                    direction = "↑" if mean_lfc > 0 else "↓"
                    genes_str = ', '.join(sig_data['gene'].tolist())
                    print(f"    {stage}: {direction} {len(sig_data)} sig genes "
                          f"(mean LFC={mean_lfc:+.2f}) — {genes_str}")
                else:
                    n_tested = len(stage_data)
                    print(f"    {stage}: no significant genes ({n_tested} tested)")

print("\nSaved exhaustion_staged_all_contrasts.csv")


In [ ]:
print(f"\n{'=' * 70}")
print("Exhaustion signature detail across all contrasts")
print("=" * 70)

exh_detail = []
for key in sorted(de_results.keys()):
    df = de_results[key]
    for gene in EXHAUSTION_SIGNATURE:
        if gene in df.index:
            row = df.loc[gene]
            exh_detail.append({
                'contrast': key,
                'gene': gene,
                'log2FoldChange': row['log2FoldChange'],
                'padj': row['padj'],
                'baseMean': row.get('baseMean', np.nan),
                'significant': row['padj'] < PADJ_THRESH if pd.notna(row['padj']) else False,
            })

exh_df = pd.DataFrame(exh_detail)
if len(exh_df) > 0:
    exh_df.to_csv(os.path.join(OUTPUT_DIR, 'exhaustion_signature_all_contrasts.csv'), index=False)

    # Pivot for heatmap
    exh_pivot = exh_df.pivot_table(index='gene', columns='contrast',
                                    values='log2FoldChange', aggfunc='first')

    # Filter to priority contrasts
    priority_cols = [c for c in exh_pivot.columns if 'Approach_B' in c and 'CT' in c]
    if len(priority_cols) > 0:
        exh_sub = exh_pivot[priority_cols].dropna(how='all')

        if len(exh_sub) > 0:
            # Clean column names for display
            clean_cols = [c.split('/')[-1].replace('_', ' ') for c in priority_cols]

            fig, ax = plt.subplots(figsize=(max(8, len(priority_cols) * 2.5),
                                           max(4, len(exh_sub) * 0.4)),
                                  facecolor='white')
            vmax = max(abs(exh_sub.min().min()), abs(exh_sub.max().max()))
            vmax = min(vmax, 3)

            sns.heatmap(exh_sub, cmap='RdBu_r', center=0, vmin=-vmax, vmax=vmax,
                       annot=True, fmt='.2f', linewidths=0.5, linecolor='white',
                       ax=ax, cbar_kws={'label': 'log2FC', 'shrink': 0.8},
                       xticklabels=clean_cols)
            ax.set_title('T-cell Exhaustion Signature — log2FC in Immune Niches (CT)',
                        fontsize=13, fontweight='bold')
            ax.set_ylabel('')
            plt.tight_layout()
            plt.savefig(os.path.join(OUTPUT_DIR, 'exhaustion_heatmap_immune_niches.png'),
                       dpi=300, bbox_inches='tight')
            plt.show()

    print("\nSaved exhaustion_signature_all_contrasts.csv")

## 10. SAVE SUMMARY

In [ ]:
print(f"\n{'=' * 70}")
print("SUMMARY")
print("=" * 70)

print(f"\nAll outputs saved to: {OUTPUT_DIR}")
print(f"  - signature_overlap_results.csv")
print(f"  - exhaustion_signature_all_contrasts.csv")
print(f"  - cross_niche_DE_genes.csv")
print(f"  - all_enrichment_results.csv (if gseapy available)")
print(f"  - Signature heatmaps (per signature)")
print(f"  - Enhanced volcano plots")
print(f"  - Enrichment results per contrast (enrichment/ subdirectory)")
print(f"\nFinished: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")